# Naive Bayes Classifier: From Scratch

## 1. Theory and Intuition

### What is Naive Bayes?
Naive Bayes is a probabilistic machine learning algorithm based on **Bayes' Theorem**. It is primarily used for classification problems. It is called "Naive" because it makes a strong assumption that the features are independent of each other given the class label.

### The Mathematics

**1. Bayes' Theorem:**
This theorem describes the probability of an event, based on prior knowledge of conditions that might be related to the event.

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

In the context of classification:
*   $A$ is the **Class** (e.g., "Spam" or "Not Spam").
*   $B$ is the **Data/Features** (e.g., words in an email).

So, we want to find the probability of a class $y$ given features $x$:

$$P(y|x) = \frac{P(x|y) \cdot P(y)}{P(x)}$$

**2. The "Naive" Assumption:**
To simplify the calculation of $P(x|y)$ (the likelihood), we assume that all features $x_1, x_2, ..., x_n$ are independent.

$$P(x|y) = P(x_1|y) \cdot P(x_2|y) \cdot ... \cdot P(x_n|y)$$

**3. The Final Formula:**
To predict the class for a new data point, we calculate the "Posterior Probability" for each possible class and choose the one with the highest score.

$$\hat{y} = \arg\max_{y} \left( P(y) \prod_{i=1}^{n} P(x_i|y) \right)$$

*   **$P(y)$**: Prior Probability (How common is the class in our data?)
*   **$P(x_i|y)$**: Likelihood (How often does feature $x_i$ appear in class $y$?)

---

## 2. Visual Explanation with Handmade Data

Let's look at a simple dataset to predict if we should **Play Tennis** based on the weather.

### Our Dataset (Handmade)

| Outlook | Temperature | Humidity | Windy | Play Tennis |
| :--- | :--- | :--- | :--- | :--- |
| Sunny | Hot | High | Weak | **No** |
| Sunny | Hot | High | Strong | **No** |
| Overcast | Hot | High | Weak | **Yes** |
| Rain | Mild | High | Weak | **Yes** |
| Rain | Cool | Normal | Weak | **Yes** |
| Rain | Cool | Normal | Strong | **No** |
| Overcast | Cool | Normal | Strong | **Yes** |
| Sunny | Mild | High | Weak | **No** |
| Sunny | Cool | Normal | Weak | **Yes** |
| Rain | Mild | Normal | Weak | **Yes** |
| Sunny | Mild | Normal | Strong | **Yes** |
| Overcast | Mild | High | Strong | **Yes** |
| Overcast | Hot | Normal | Weak | **Yes** |
| Rain | Mild | High | Strong | **No** |

### Step-by-Step Calculation for a New Sample

**Question:** Should we play tennis if the weather is: **Outlook=Sunny, Temperature=Cool, Humidity=High, Windy=Strong**?

We need to calculate probabilities for **Yes** and **No**.

#### A. Calculate Prior Probabilities $P(Play)$

*   Total rows = 14
*   **Yes** count = 9
*   **No** count = 5

$$P(Yes) = \frac{9}{14} \approx 0.64$$
$$P(No) = \frac{5}{14} \approx 0.36$$

#### B. Calculate Likelihoods $P(Feature|Class)$

**For Class = "No":**
1.  $P(Outlook=Sunny | No)$: How many times is it Sunny when we don't play? (3 out of 5) $\rightarrow \frac{3}{5}$
2.  $P(Temperature=Cool | No)$: How many times is it Cool when we don't play? (1 out of 5) $\rightarrow \frac{1}{5}$
3.  $P(Humidity=High | No)$: How many times is High humidity when we don't play? (4 out of 5) $\rightarrow \frac{4}{5}$
4.  $P(Windy=Strong | No)$: How many times is Strong wind when we don't play? (3 out of 5) $\rightarrow \frac{3}{5}$

**Probability of "No":**
$$P(No) \times P(Sunny|No) \times P(Cool|No) \times P(High|No) \times P(Strong|No)$$
$$= \frac{5}{14} \times \frac{3}{5} \times \frac{1}{5} \times \frac{4}{5} \times \frac{3}{5} \approx 0.0206$$

**For Class = "Yes":**
1.  $P(Outlook=Sunny | Yes)$: (2 out of 9) $\rightarrow \frac{2}{9}$
2.  $P(Temperature=Cool | Yes)$: (3 out of 9) $\rightarrow \frac{3}{9}$
3.  $P(Humidity=High | Yes)$: (3 out of 9) $\rightarrow \frac{3}{9}$
4.  $P(Windy=Strong | Yes)$: (3 out of 9) $\rightarrow \frac{3}{9}$

**Probability of "Yes":**
$$P(Yes) \times P(Sunny|Yes) \times P(Cool|Yes) \times P(High|Yes) \times P(Strong|Yes)$$
$$= \frac{9}{14} \times \frac{2}{9} \times \frac{3}{9} \times \frac{3}{9} \times \frac{3}{9} \approx 0.0143$$

#### C. Compare

*   $P(No | Data) \propto 0.0206$
*   $P(Yes | Data) \propto 0.0143$

Since $0.0206 > 0.0143$, the algorithm predicts **"No"**, we should not play tennis.

---

## 3. Python Implementation (From Scratch)

Now, let's write the code to implement this logic automatically.

In [7]:
import pandas as pd
import numpy as np

# 1. Create the Handmade Dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 
                'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 
                    'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 
                 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Windy': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 
              'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 
                   'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
print("Dataset:")
print(df)

Dataset:
     Outlook Temperature Humidity   Windy PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No


# 2. Define the Naive Bayes Class

In [8]:
class NaiveBayesClassifier:
    def __init__(self):
        self.prior = {}
        self.likelihood = {}
        self.classes = None

    def fit(self, X, y):
        """
        Train the model by calculating Prior and Likelihood probabilities.
        X: DataFrame of features
        y: Series of target labels
        """
        self.classes = y.unique()
        features = X.columns
        
        # Calculate Priors P(Class)
        for cls in self.classes:
            self.prior[cls] = len(y[y == cls]) / len(y)
            
        # Calculate Likelihoods P(Feature|Class)
        for feature in features:
            self.likelihood[feature] = {}
            for cls in self.classes:
                # Filter data for the specific class
                feature_data = X[y == cls][feature]
                
                # Calculate probability for each unique value of the feature
                value_counts = feature_data.value_counts(normalize=True)
                self.likelihood[feature][cls] = value_counts.to_dict()

    def predict(self, X_new):
        """
        Predict class for a new sample (dictionary or DataFrame row)
        """
        results = {}
        
        for cls in self.classes:
            # Start with the Prior Probability
            prob = self.prior[cls]
            
            # Multiply by Likelihood of each feature value
            for feature, value in X_new.items():
                if value in self.likelihood[feature][cls]:
                    prob *= self.likelihood[feature][cls][value]
                else:
                    # Handle unseen values (Laplace smoothing could be added here)
                    prob *= 0 
                    
            results[cls] = prob
            
        # Return the class with the highest probability
        return max(results.items(), key=lambda x: x[1])[0], results

# 3. Train the Model
X = df.drop('PlayTennis', axis=1)
y = df['PlayTennis']

model = NaiveBayesClassifier()
model.fit(X, y)

print("Training Complete.\n")
print(f"Calculated Priors: {model.prior}")


Training Complete.

Calculated Priors: {'No': 0.35714285714285715, 'Yes': 0.6428571428571429}


# 4. Test with a New Sample


In [9]:
# Let's test the example we calculated manually:
# Outlook=Sunny, Temperature=Cool, Humidity=High, Windy=Strong

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Windy': 'Strong'
}

prediction, probabilities = model.predict(new_sample)

print(f"Input Data: {new_sample}")
print("-" * 30)
print(f"Probabilities: {probabilities}")
print("-" * 30)
print(f"Final Prediction: {prediction}")

Input Data: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Windy': 'Strong'}
------------------------------
Probabilities: {'No': 0.02057142857142857, 'Yes': 0.005291005291005291}
------------------------------
Final Prediction: No


### Summary
In this notebook, we:
1.  Learned the theory behind **Bayes' Theorem**.
2.  Manually calculated probabilities using a handmade dataset.
3.  Implemented a **Naive Bayes Classifier** in Python without using external libraries like Scikit-Learn.

This code can handle any categorical dataset. If you want to proceed to the next step (like handling numerical data or Laplace Smoothing), please let me know